In [ ]:
# Step 1: Setup and data 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, confusion_matrix
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

df = pd.read_csv("../data/processed/train_merged.csv")
print(df.shape)

(307511, 134)


In [2]:
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
X = X.replace([np.inf, -np.inf], np.nan)

# text columns -> category type, lightgbm can use these directly
cat_cols = X.select_dtypes(include="object").columns
for c in cat_cols:
    X[c] = X[c].astype("category")

print(X.shape, len(cat_cols))

C:\Users\HP\AppData\Local\Temp\ipykernel_5124\669988290.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


(307511, 132) 16


In [3]:
# Step 2: Same split as the baseline
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_val.shape)
print(y_val.mean())

(246008, 132) (61503, 132)
0.08072776937710356


In [4]:
# a small part of train is kept just for early stopping,
# so the validation set stays untouched until the final evaluation
X_tr, X_es, y_tr, y_es = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
)

In [5]:
# Step 3: Train
model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_es, y_es)],
    eval_metric="auc",
    callbacks=[early_stopping(100), log_evaluation(100)]
)

print("best number of trees:", model.best_iteration_)

c:\Users\HP\Desktop\Projects\risklens-explainable-credit-risk\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 17874, number of negative: 203533
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.099272 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13457
[LightGBM] [Info] Number of data points in the train set: 221407, number of used features: 128
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.755729	valid_0's binary_logloss: 0.574911
[200]	valid_0's auc: 0.758166	valid_0's binary_logloss: 0.557729
Early stopping, best iteration is:
[167]	valid_0's auc: 0.758316	valid_0's binary_logloss: 0.562343
best number of trees: 167


In [6]:
# Step 4: Evaluate (same metrics as the baseline)
probs = model.predict_proba(X_val)[:, 1]

print("ROC AUC:", roc_auc_score(y_val, probs))
print("PR AUC:", average_precision_score(y_val, probs))

preds = (probs > 0.5).astype(int)
print("recall:", recall_score(y_val, preds))
print("precision:", precision_score(y_val, preds))
print(confusion_matrix(y_val, preds))

ROC AUC: 0.7634728393601152
PR AUC: 0.25806049952491167
recall: 0.6737160120845922
precision: 0.1737752610525222
[[40634 15904]
 [ 1620  3345]]


In [7]:
#Step 5: Feature importance
imp = pd.Series(
    model.booster_.feature_importance(importance_type="gain"),
    index=X.columns
).sort_values(ascending=False)

print(imp.head(15))

EXT_SOURCE_3           182402.486420
EXT_SOURCE_2           141250.321926
ORGANIZATION_TYPE       77094.845118
EXT_SOURCE_1            50196.148817
DAYS_EMPLOYED           21677.863119
AMT_CREDIT              17793.261574
OCCUPATION_TYPE         15437.782642
prev_refused_rate       15398.276045
AMT_GOODS_PRICE         13262.328070
OWN_CAR_AGE             11939.192574
NAME_EDUCATION_TYPE     11728.880318
AMT_ANNUITY             11115.642710
CODE_GENDER             10973.251244
bureau_debt_sum         10331.674768
bureau_credit_sum        8668.871172
dtype: float64


In [9]:
#Step 6: Save the model for the API later
import joblib
joblib.dump(model, "../models/lgbm_model.joblib")

['../models/lgbm_model.joblib']